In [ ]:
"""Python translation of ``example_processFDN.m`` including processing call."""

from __future__ import annotations

from pathlib import Path
import subprocess

from importlib.resources import files

import numpy as np
from scipy.io import wavfile

from pyFDN.auxiliary.matrix_delay_approximation import matrix_delay_approximation
from pyFDN.auxiliary.one_pole_absorption import one_pole_absorption
from pyFDN.auxiliary.process_fdn import process_fdn
from pyFDN.auxiliary.ztf import ZTF
from pyFDN.generate.construct_velvet_feedback_matrix import construct_velvet_feedback_matrix
from pyFDN.generate.random_matrix_shift import random_matrix_shift


def main() -> None:
    audio_resource = files("pyFDN.audio") / "synth_dry.wav"
    with audio_resource.open("rb") as audio_file:
        fs, audio = wavfile.read(audio_file)

    if audio.ndim > 1:
        x = audio[:, 0]
    else:
        x = audio

    if np.issubdtype(x.dtype, np.integer):
        max_val = np.iinfo(x.dtype).max
        x = x.astype(np.float64) / max_val
    else:
        x = x.astype(np.float64)

    x = np.concatenate([x, np.zeros(2 * fs, dtype=np.float64)])

    rng = np.random.default_rng()

    N = 8
    num_input = 1
    num_output = 2

    input_gain = np.ones((N, num_input))
    output_gain = rng.random((num_output, N))
    direct = np.zeros((num_output, num_input))

    delays = rng.integers(500, 2001, size=N)
    number_of_stages = 2
    sparsity = 3
    max_shift = 30

    feedback_matrix, rev_feedback_matrix = construct_velvet_feedback_matrix(N, number_of_stages, sparsity)
    feedback_matrix, rev_feedback_matrix, _, _ = random_matrix_shift(
        max_shift, feedback_matrix, rev_feedback_matrix
    )

    approximation, approximation_error = matrix_delay_approximation(feedback_matrix)

    rt_dc = 2.0
    rt_ny = 0.5
    b_abs, a_abs = one_pole_absorption(rt_dc, rt_ny, delays + approximation, fs=float(fs))
    b_padded = np.zeros_like(a_abs)
    b_padded[:, :, 0] = b_abs[:, :, 0]
    z_absorption = ZTF(b_padded, a_abs, is_diagonal=True)

    output = process_fdn(
        x[:, np.newaxis],
        delays,
        feedback_matrix,
        input_gain,
        output_gain,
        direct,
        absorption_filters=z_absorption,
    )

    output = np.asarray(output, dtype=np.float64)
    if output.ndim == 1:
        output = output[:, np.newaxis]

    peak = np.max(np.abs(output))
    if peak > 0:
        output = output / peak * 0.99

    outfile_path = Path.cwd() / "example_process_fdn_output.wav"
    wavfile.write(outfile_path, fs, np.int16(output * 32767.0))

    try:
        subprocess.run(["afplay", str(outfile_path)], check=False)
    except FileNotFoundError:
        pass

    print("Input length:", x.shape[0])
    print("Output shape:", output.shape)
    print("Approximation mean abs error:", np.mean(np.abs(approximation_error)))
    print("Output RMS:", np.sqrt(np.mean(np.asarray(output) ** 2)))


if __name__ == "__main__":
    main()
